In [2]:
from pypdf import PdfReader
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import re
from rapidfuzz import fuzz
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from google import genai

### Reading the PDF and Extracting the text

In [3]:
pdf = []
try:
    reader = PdfReader('./PDFs/bom.pdf')
    for page in reader.pages:
         pdf.append(page.extract_text())

    print(f"Total Pages: {len(pdf)}")

except Exception as exc:
    # TO Catch error 
    print("Remaining")

Total Pages: 224


### Utility Function 

In [4]:

def normalize_text(text: str) -> str:
    """ 
    Cleans extracted PDF text for better chunking and retrieval. 
    """
    # Step 1: Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # # Step 2: Remove repeated book title/header
    # text = re.sub("Basics of Mathematics\s+\d+\s*", "", text)

    # # Step 3: Remove standalone page numbers
    # text = re.sub(r"\n\s*\d+\s*\n", "\n", text)

    # Step 4: Replace multiple space with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Step 5: Replace multiple blank lines with at most two
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text


def compute_line_similarity(pdf, n_lines=3, similarity_fn=fuzz.ratio):
    """
    Compare the first and last `n_lines` of consecutive PDF pages.

    Parameters
    ----------
    pdf : list[str]
        List containing page-wise extracted text.

    n_lines : int
        Number of header/footer candidate lines.

    similarity_fn : function
        RapidFuzz similarity function.
        Examples:
            fuzz.ratio
            fuzz.partial_ratio
            fuzz.token_sort_ratio
            fuzz.token_set_ratio
            fuzz.WRatio

    Returns
    -------
    list[dict]
        Similarity information for every consecutive page pair.
    """

    results = []

    for page_no, (current, nxt) in enumerate(zip(pdf, pdf[1:]), start=1):

        current_lines = [line.strip() for line in current.splitlines() if line.strip()]
        next_lines = [line.strip() for line in nxt.splitlines() if line.strip()]

        current_header = current_lines[:n_lines]
        next_header = next_lines[:n_lines]

        current_footer = current_lines[-n_lines:]
        next_footer = next_lines[-n_lines:]

        page_result = {
            "pages": (page_no, page_no + 1),
            "header_scores": [],
            "footer_scores": []
        }

        # -------- Header Comparison --------
        for idx, (line1, line2) in enumerate(zip(current_header, next_header), start=1):

            score = similarity_fn(line1, line2)

            page_result["header_scores"].append({
                "line": idx,
                "text1": line1,
                "text2": line2,
                "score": score
            })

        # -------- Footer Comparison --------
        for idx, (line1, line2) in enumerate(zip(current_footer, next_footer), start=1):

            score = similarity_fn(line1, line2)

            page_result["footer_scores"].append({
                "line": idx,
                "text1": line1,
                "text2": line2,
                "score": score
            })

        results.append(page_result)

    return results





def identify_boilerplate(
    similarity_df,
    mean_threshold=90,
    std_threshold=10
):
    """
    Identify repeated boilerplate lines using similarity statistics.

    Parameters
    ----------
    similarity_df : pandas.DataFrame
        DataFrame containing:
            region
            line
            score

    mean_threshold : float
        Minimum average similarity required.

    std_threshold : float
        Maximum allowed standard deviation.

    Returns
    -------
    pandas.DataFrame
        Summary statistics and classification.
    """

    statistics = (
        similarity_df
        .groupby(["region", "line"])["score"]
        .agg(
            mean_score="mean",
            std_score="std"
        )
        .reset_index()
    )

    statistics["is_boilerplate"] = (
        (statistics["mean_score"] >= mean_threshold)
        &
        (statistics["std_score"] <= std_threshold)
    )

    return statistics

### Text Preprocessing

##### Cell 1: Regex Based Preprocessing

In [8]:
# Task 1: Normalize the text as much as possible
normalized_pdf = []
i = 0

for page in pdf:
    normalized_page = normalize_text(page)
    normalized_pdf.append(normalized_page)    


# Comparing The cleaned PDF vs Original PDF
for i in range(0,5):
    print("---------------------------------------")
    print("------------- Original PDF ------------") 
    print(pdf[i])
    print("---------------------------------------\n")

    print("---------------------------------------")
    print("------------- Cleaned PDF ------------") 
    print(normalized_pdf[i])
    print("---------------------------------------\n")

---------------------------------------
------------- Original PDF ------------
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 
7. Simplifications 58 
8. Ratio and Proportion 69 
9. Chain rule 
74 
10. Averages 82 
11. Percentages 91 
12. Profit and Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
 
---------------------------------------

---------------------------------------
------------- Cleaned PDF ------------
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quant

#### Cell 2: Statistics Based Preprocessing

In [9]:
similar_line_pdf = compute_line_similarity(pdf, 3)

In [10]:
suspect_similarity_dataset = pd.DataFrame(columns=['text1','text2', 'line', 'page_from', 'page_to', 'region',  'score'])

for page in similar_line_pdf: 
    for header in page['header_scores']:
        new_row = {
            'page_from': page['pages'][0],
            'page_to': page['pages'][1],
            'region': 'header',
            'line': header['line'],
            'text1': header['text1'],
            'text2': header['text2'],
            'score': header['score']
        }
        suspect_similarity_dataset.loc[len(suspect_similarity_dataset)] = new_row
        
    for footer in page['footer_scores']:
        new_row = {
            'page_from': page['pages'][0],
            'page_to': page['pages'][1],
            'region': 'footer',
            'line': footer['line'],
            'text1': footer['text1'],
            'text2': footer['text2'],
            'score': footer['score']
        }
        suspect_similarity_dataset.loc[len(suspect_similarity_dataset)] = new_row


suspect_similarity_dataset.head(10)


,text1,text2,line,page_from,page_to,region,score
0,Basics of Mathematics,Basics of Mathematics ...,1,1,2,header,26.415094
1,Sr. No. Topic name Page number,Section A,2,1,2,header,20.512821
2,1. Qualitative and Quantitative Analysis 2,A.1 Introduction to Qualitative and,3,1,2,header,49.350649
3,19. Series 205,experience (UX),1,1,2,footer,34.482759
4,20. Problems on Ages 210,Researching competitors’ website pages to iden...,2,1,2,footer,24.347826
5,21. Discounts – its types and Analysis 216,propositions,3,1,2,footer,22.222222
6,Basics of Mathematics ...,Basics of Mathematics ...,1,2,3,header,99.275362
7,Section A,Interpreting customer interviews and survey re...,2,2,3,header,14.736842
8,A.1 Introduction to Qualitative and,setting the direction for new product or featu...,3,2,3,header,37.500000
9,experience (UX),This might look like analyzing daily content s...,1,2,3,footer,16.513761


In [11]:
"""
Computing the mean and the Std
"""

# Headers
suspect_similarity_mean = suspect_similarity_dataset.groupby(['region', 'line'])['score'].mean()
suspect_similarity_std = suspect_similarity_dataset.groupby(['region', 'line'])['score'].std()


print("::::::::::::::: Line Statistics :::::::::::::::")
print("Mean :::>")
print(suspect_similarity_mean)

print("\n\n")
print("Std :::>")
print(suspect_similarity_std)

::::::::::::::: Line Statistics :::::::::::::::
Mean :::>
region  line
footer  1       95.856193
        2       35.873401
        3       28.987282
header  1       98.873913
        2       25.551731
        3       30.893231
Name: score, dtype: float64



Std :::>
region  line
footer  1       15.738222
        2       21.741000
        3       10.501335
header  1        4.879473
        2        8.896255
        3       13.263025
Name: score, dtype: float64


In [12]:
data = identify_boilerplate(similarity_df=suspect_similarity_dataset)


filtered_data = data[data['is_boilerplate'] == True]

print(filtered_data)


   region  line  mean_score  std_score  is_boilerplate
3  header     1   98.873913   4.879473            True


In [13]:
text = "".join(normalized_pdf)

text

"Basics of Mathematics \nSr. No. Topic name Page number \n1. Qualitative and Quantitative Analysis 2 \n2. Number and Number System, problems on Numbers 7 \n3. Fractions, Decimals and Numbers 32 \n4. Divisibility and Divisibility Test 39 \n5. HCF and LCM 44 \n6. Squares, Cubes, Square-roots & Cube-roots 51 \n7. Simplifications 58 \n8. Ratio and Proportion 69 \n9. Chain rule \n74 \n10. Averages 82 \n11. Percentages 91 \n12. Profit and Loss 101 \n13. Simple and Compound Interest 110 \n14. Heights, Distances & Basic Trigonometry 135 \n15. Time, Work and Wages 142 \n16. Time and Distances 162 \n17. Trains, Boats and Streams 176 \n18. Clock 197 \n19. Series 205 \n20. Problems on Ages 210 \n21. Discounts – its types and Analysis 216 \n \n \n  Basics of Mathematics 2 \n \nSection A \nA.1 Introduction to Qualitative and \nQuantitative Analysis \nQualitative Data Analysis \nQualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting \nqualitative data—non-numeric, c

### Chunking the PDF
:-- Using recursive chunking

In [46]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        " ",
        ".",
        ",",
        "\u200b",  # Zero-width space
        "\uff0c",  # Fullwidth comma
        "\u3001",  # Ideographic comma
        "\uff0e",  # Fullwidth full stop
        "\u3002",  # Ideographic full stop
        "",
    ],
)

all_splits = text_splitter.split_text(text)

print(f"Split PDF text into {len(all_splits)} sub-documents.")


Split PDF text into 30 sub-documents.


In [ ]:
for i, chunk in enumerate(all_splits):
    print(f"\n--- Chunk {i+1} ---")
    print(f"\n--- First half of chunk {i+1} ---")
    print(chunk[:300])
    print(f"\n--- Second half of chunk {i+1} ---")
    print(chunk[300:])
    print(f"\nLength: {len(chunk)}")

    if i == 2:
        break


--- Chunk 1 ---

--- First half of chunk 1 ---
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 


--- Second half of chunk 1 ---
7. Simplifications 58 
8. Ratio and Proportion 69 
9. Chain rule 
74 
10. Averages 82 
11. Percentages 91 
12. Profit and Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
       Basics of Mathematics                                                                                                                    2 
 
Section A 
A.1 Introduction to Qualitative and 
Qua

### Embeddings

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(all_splits)

print(f"Each embedding has {embeddings.shape[1]} dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Each embedding has 384 dimensions


In [ ]:
similarities = cosine_similarity(embeddings)
print("Similarity matrix:")
print(similarities)
# You'll see sentences 1, 2, and 4 are more similar to each other


Similarity matrix:
[[0.9999999  0.27995884 0.32016426 ... 0.5082781  0.4807489  0.4872148 ]
 [0.27995884 0.9999999  0.76331556 ... 0.1517172  0.17621809 0.12574951]
 [0.32016426 0.76331556 1.         ... 0.24193653 0.25580803 0.23576777]
 ...
 [0.5082781  0.1517172  0.24193653 ... 1.         0.9651926  0.90530217]
 [0.4807489  0.17621809 0.25580803 ... 0.9651926  1.         0.88975585]
 [0.4872148  0.12574951 0.23576777 ... 0.90530217 0.88975585 1.        ]]


## Storing into the Database

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

In [ ]:
print(index.ntotal)

66


In [ ]:
# query = "What is qualitative data analysis?"
query = "What is Qualitative Analysis?"

query_embedding = model.encode([query])


## Searching into Database

In [ ]:
k = 3

distances, indices = index.search(
    query_embedding,
    k
)

In [ ]:
retrived_text = []
for idx in indices[0]:
    retrived_text.append(all_splits[idx])

retrived_text

["Section A \nA.1 Introduction to Qualitative and \nQuantitative Analysis \nQualitative Data Analysis \nQualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting \nqualitative data—non-numeric, conceptual information and user feedback—to capture \nthemes and patterns, answer research questions, and identify actions to take to improve your \nproduct or website. \nHere are five methods of qualitative data analysis to help you make sense of the data you've \ncollected through customer interviews, surveys, and feedback: \n1. Content analysis \n2. Thematic analysis \n3. Narrative analysis \n4. Grounded theory analysis \n5. Discourse analysis \n1. Content analysis \nContent analysis is a research method that examines and quantifies the presence of certain \nwords, subjects, and concepts in text, image, video, or audio messages. The \nmethod transforms qualitative input into quantitative data to help you make reliable",
 'In a business context, the method is pr

## Response Generation using LLM
---

### Cell 1
#### Responsibility : - Configure LLM Model

In [ ]:
# Client Configuration
API_KEY = 'AQ.Ab8RN6LdZl8pxdRhnCTAMsjCvQuzFv1CRJt3JGnqdkmhUR-GhQ'

client = genai.Client(api_key=API_KEY)

### Cell 2
#### Responsibility : - Prompt Builder

In [ ]:
context = "\n\n".join(retrived_text)

prompt = f"""
You are an AI assistant.

Answer ONLY using the provided context.

If the answer is not available,
say "I don't know."

Context:
{context}

Query:
{query}
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents= prompt
)

In [ ]:
print("response")
print(response.text)

response
Qualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting qualitative data—non-numeric, conceptual information and user feedback—to capture themes and patterns, answer research questions, and identify actions to take to improve your product or website.
